In [ ]:
from pathlib import Path
import random
from typing import List
import cv2
import numpy as np
import yaml

from tqdm import tqdm

from typing import List, Tuple

In [ ]:
### GLOBAL SETTINGS ###

INPUT_SIZE = 800
SEED = 42

SCAN_DIR = Path('../data/selection/scans')
MASK_DIR = Path('../data/selection/masks')

PROCESSED_DIR = Path(f'../data/processed_{INPUT_SIZE}')

IMAGES_TRAIN = PROCESSED_DIR / 'images' / 'train'
IMAGES_VAL = PROCESSED_DIR / 'images' / 'val'
IMAGES_TEST = PROCESSED_DIR / 'images' / 'test'

LABELS_TRAIN = PROCESSED_DIR / 'labels' / 'train'
LABELS_VAL = PROCESSED_DIR / 'labels' / 'val'
LABELS_TEST = PROCESSED_DIR / 'labels' / 'test'

YAML_CONFIG_DIR = Path('../data/YOLO_CONFIGS')

random.seed(SEED)

In [3]:
# Create output directories
for p in [IMAGES_TRAIN, IMAGES_VAL, IMAGES_TEST, LABELS_TRAIN, LABELS_VAL, LABELS_TEST]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
def mask_to_bbox(mask_path:Path) -> List:

    """
        Read a grayscale mask image and return a list of axis-aligned bounding boxes
        for each connected component (foreground object).

        Args:
            mask_path: Path to the mask image (typically uint8 grayscale).
                    Foreground (object) should be dark (close to 0),
                    background bright (close to 255) — common in segmentation masks.

        Returns:
            List of bounding boxes in [x_min, y_min, x_max, y_max] format (inclusive).

        Raises:
            FileNotFoundError: If the mask file cannot be read.
    """

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f'Mask not found: {mask_path}')

    # binarize to 0 and 255
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY_INV)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    bounding_boxes = []
    for cnt in contours:
        #area = cv2.contourArea(cnt)
        x, y, w, h = cv2.boundingRect(cnt)
        bounding_boxes.append([x, y, x + w, y + h])
    
    return bounding_boxes

def resize_and_pad(img:np.ndarray, target_size:int=INPUT_SIZE) -> Tuple[np.ndarray, float, Tuple[int, int]]:
    
    """
    Resize image to fit within target_size × target_size while preserving aspect ratio,
    then center-pad with white borders (value=255) to make it exactly square.

    Args:
        img: Input image as numpy array (H, W) or (H, W, C), uint8 recommended
        target_size: Desired output size (both width and height). Defaults to INPUT_SIZE.

    Returns:
        padded_img (np.ndarray): Final square image of shape (target_size, target_size, C)
        scale (float): Scaling factor applied = target_size / max(original_h, original_w)
        (pad_left, pad_top) (Tuple[int, int]): Padding added on left and top sides
    """

    h, w = img.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    pad_w = target_size - new_w
    pad_h = target_size - new_h
    left = pad_w // 2
    top = pad_h // 2
    right = pad_w - left
    bottom = pad_h - top
    
    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=255)  # white padding
    return padded, scale, (left, top)

def scale_bbox(bbox:List, scale:float, pad_left:int, pad_top:int) -> List:
    
    """
    Transform a bounding box from original image coordinates
    to the resized + padded (letterboxed) image coordinates.

    This is the forward transformation used before inference
    (opposite of what you do when drawing predictions back).

    Args:
        bbox: Input bounding box as [x_min, y_min, x_max, y_max]
        scale: Scaling factor (same as returned by resize_and_pad)
        pad_left: Left padding added during letterboxing
        pad_top: Top padding added during letterboxing

    Returns:
        Scaled and shifted bbox as (x_min, y_min, x_max, y_max) with float coords
        (useful for submission formats or non-integer models)
    """

    x_min, y_min, x_max, y_max = bbox
    x_min = x_min * scale + pad_left
    y_min = y_min * scale + pad_top
    x_max = x_max * scale + pad_left
    y_max = y_max * scale + pad_top
    return [x_min, y_min, x_max, y_max]

def bbox_to_yolo(bbox:List, img_w:int, img_h:int) -> List:

    """
    Convert bounding box from [x_min, y_min, x_max, y_max] (pixel coordinates)
    to YOLO format: [class_id, x_center_norm, y_center_norm, width_norm, height_norm]

    All spatial values are normalized to [0, 1] relative to image width/height.

    Args:
        bbox: Bounding box as [x_min, y_min, x_max, y_max]
        img_w: Image width in pixels
        img_h: Image height in pixels
        class_id: Class index (default: 0, for single-class models like stamp detection)

    Returns:
        List: [class_id, x_center, y_center, w, h]
    """

    x_min, y_min, x_max, y_max = bbox
    x_center = (x_min + x_max) / 2 / img_w
    y_center = (y_min + y_max) / 2 / img_h
    w = (x_max - x_min) / img_w
    h = (y_max - y_min) / img_h
    return [0, x_center, y_center, w, h]  # class_id = 0

def save_yolo_label(bboxes_yolo:List, label_path:Path) -> None:
    """Save list of YOLO lines to .txt"""
    with open(label_path, 'w') as f:
        for box in bboxes_yolo:
            line = ' '.join(map(str, box))
            f.write(line + '\n')

In [ ]:
def process_pair(scan_path:Path, mask_path:Path, img_out_dir:Path, label_out_dir:Path) -> None:
    """Process one image-mask pair"""
    img = cv2.imread(str(scan_path))
    if img is None:
        raise FileNotFoundError(f"Image not found: {scan_path}")
    
    # Extract bboxes from mask
    raw_bboxes = mask_to_bbox(mask_path)
    
    # Resize and pad image
    padded_img, scale, (pad_left, pad_top) = resize_and_pad(img, INPUT_SIZE)
    
    # Scale bboxes
    scaled_bboxes = [
        scale_bbox(bbox, scale, pad_left, pad_top)
        for bbox in raw_bboxes
    ]
    
    # Convert to YOLO format
    yolo_bboxes = [
        bbox_to_yolo(bbox, INPUT_SIZE, INPUT_SIZE)
        for bbox in scaled_bboxes
    ]
    
    # Save image
    img_name = scan_path.stem + '.jpg'  # save as JPG to reduce size
    cv2.imwrite(str(img_out_dir / img_name), padded_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    
    # Save label
    label_path = label_out_dir / (scan_path.stem + '.txt')
    save_yolo_label(yolo_bboxes, label_path)

def create_data_yaml(config_name:str = 'data_X.yaml', input_size=INPUT_SIZE) -> None:
    data = {
        'train': str(IMAGES_TRAIN.resolve()),
        'val': str(IMAGES_VAL.resolve()),
        'nc': 1,
        'names': ['stamp']
    }
    yaml_path = YAML_CONFIG_DIR / config_name.replace('X',str(input_size))
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)
    print(f"data.yaml created at: {yaml_path}")

In [6]:
scan_files = sorted([f for f in SCAN_DIR.iterdir() if f.suffix.lower() == '.png'])
mask_files = {f.name.replace('-gt',''): f for f in MASK_DIR.iterdir() if f.suffix.lower() == '.png'}

In [7]:
pairs = []
for scan_path in scan_files:
    mask_path = mask_files.get(scan_path.name)
    if mask_path is None:
        print(f"Warning: No mask for {scan_path.name}, skipping.")
        continue
    pairs.append((scan_path, mask_path))

In [8]:
# preparation of training / validation / test sets

random.shuffle(pairs)
split_idx = int(len(pairs) * 0.8)
train_pairs = pairs[:split_idx]
val_pairs_temp = pairs[split_idx:]
split_idx = int(len(val_pairs_temp) * 0.8)
val_pairs = val_pairs_temp[:split_idx]
test_pairs = val_pairs_temp[split_idx:]


print(f"Total pairs: {len(pairs)} | Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)} ")

Total pairs: 400 | Train: 320 | Val: 64 | Test: 16 


In [9]:
# Process train set
for scan_path, mask_path in tqdm(train_pairs, desc="Processing train"):
    process_pair(scan_path, mask_path, IMAGES_TRAIN, LABELS_TRAIN)

# Process val set
for scan_path, mask_path in tqdm(val_pairs, desc="Processing val"):
    process_pair(scan_path, mask_path, IMAGES_VAL, LABELS_VAL)

# Process test set
for scan_path, mask_path in tqdm(test_pairs, desc="Processing test"):
    process_pair(scan_path, mask_path, IMAGES_TEST, LABELS_TEST)

Processing test: 100%|██████████| 16/16 [00:02<00:00,  5.46it/s]


In [11]:
# create cofiguration for YOLO training

create_data_yaml(config_name='data_1280.yaml')

data.yaml created at: ../data/YOLO_CONFIGS/data_1280.yaml
